# Fuel Cost-Shock Forecaster — Multi-Agent Setup

Adapted from the [`energy_oil_forecasting`](../energy_oil_forecasting/) (WTI) reference implementation. Full objectives: [`../../fuel-cost-shock-forecaster/Fuel_Cost_Shock_Forecaster_Objectives_and_Requirements.md`](../../fuel-cost-shock-forecaster/Fuel_Cost_Shock_Forecaster_Objectives_and_Requirements.md).

**Bootcamp convention: everything lives in this notebook.** No `data.py` / `tasks.py` / `news_agent/` / `forecaster_agent/` package modules — the Commodity-Data ingestion, the Geopolitical/News Agent, the Forecaster Agent, and the task specs are all defined inline below so the whole multi-agent pipeline runs top-to-bottom from this one file.

**Architecture: 3 agents, no Adjudicator.** The Adjudicator agent from the original objectives doc was dropped — there's no existing pattern in this repo for multi-run adjudication to build on, and its "flag the dominant market drivers" job is already covered by `DiscreteAgentForecastOutput.key_signals` / `.reasoning` on the Forecaster Agent's shock output (see the shock task spec below).

| Agent | Section | Role |
|---|---|---|
| Commodity-Data Agent | §2 | `DataService` registration: yfinance (jet-fuel proxy, WTI, USD index) + FRED (CAD/USD FX) + EIA (crude/jet-fuel spot) + the derived cost-shock event series |
| Geopolitical/News Agent | §3 | `ContextRetrievalConfig` sub-agent (`search_web` tool) with a temporal-leakage verifier |
| Forecaster Agent | §4 | Top-level `AgentConfig`; "one agent, two tasks" — binary shock (primary) + continuous trajectory (secondary) |

**Targets:**
- **Primary (binary):** P(jet-fuel proxy rises > 10% over the next 21 trading days).
- **Secondary (continuous):** jet-fuel proxy point forecast at 5/10/21 trading days.

**Jet-fuel proxy:** no liquid public jet-fuel futures contract exists, so NY Harbor ULSD / Heating Oil (`HO=F`) is used — the standard middle-distillate proxy convention (jet fuel prices as a spread off heating oil / diesel).

In [1]:
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display  # noqa: A004

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters.base import BaseAdapter
from aieng.forecasting.data.adapters.fred import FREDAdapter
from aieng.forecasting.data.adapters.yfinance import YFinanceDailyAdapter
from aieng.forecasting.data.context import ForecastContext
from aieng.forecasting.evaluation.backtest import compute_brier_score
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES
from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.methods.agentic import (
    AgentPredictor,
    ContinuousAgentForecastOutput,
    DiscreteAgentForecastOutput,
)
from aieng.forecasting.methods.agentic.agent_factory import AgentConfig, ContextRetrievalConfig
from aieng.forecasting.models import ADVANCED_MODEL, LITE_MODEL
from dotenv import load_dotenv
from pydantic import BaseModel


warnings.filterwarnings("ignore")
load_dotenv()

# ── Model selection ──────────────────────────────────────────────────────────────────────────
AGENT_MODEL = LITE_MODEL
SEARCH_MODEL = LITE_MODEL
VERIFIER_MODEL = ADVANCED_MODEL

# ── Cost-shock target definition ──────────────────────────────────────────────────
SHOCK_THRESHOLD_PCT = 0.10
SHOCK_HORIZON_DAYS = 21  # ~1 trading month
TRAJECTORY_HORIZONS = [5, 10, 21]


def naive_utc_now() -> datetime:
    """Timezone-naive current UTC time (DataService / CutoffEnforcer require this)."""
    return datetime.now(tz=timezone.utc).replace(tzinfo=None)


def repo_data_dir() -> Path:
    """Return ``data/`` at the repository root (walk up from CWD if needed)."""
    cwd = Path.cwd().resolve()
    root = cwd
    while not (root / "pyproject.toml").exists():
        if root.parent == root:
            return cwd / "data"
        root = root.parent
    data_dir = root / "data"
    data_dir.mkdir(exist_ok=True)
    return data_dir


DATA_DIR = repo_data_dir()
print(f"Data cache root: {DATA_DIR}")

ModuleNotFoundError: No module named 'pandas'

---
## 1. Commodity-Data Agent

Realised as a `DataService` registration layer, not a separate LLM call — every reference implementation in this repo (WTI, BoC, S&P 500) treats quantitative time-series data as a structured prompt payload, not something an agent fetches or summarizes. `FuelMultitaskPromptBuilder` (§4) builds the payload from the service below, fulfilling this agent's role structurally.

**Data sources (doc §6):**

| Source | Series | Key required? |
|---|---|---|
| yfinance | jet-fuel proxy `HO=F`, WTI `CL=F`, USD index `DX-Y.NYB` | No |
| FRED | `DEXCAUS` (CAD/USD spot FX) — stands in for the doc's "Bank of Canada Valet API" (no Valet adapter exists in this repo; FRED covers the same signal) | `FRED_API_KEY` (set) |
| EIA Open Data | `RWTC` (WTI Cushing spot), `EER_EPJK_PF4_RGC_DPG` (US Gulf Coast jet-fuel spot) — verified against `eia-api-swagger.yaml` | `EIA_API_KEY` (set) |

EIA inventory/stocks series (`petroleum/stoc/...`) are **not yet added** — not verified against the swagger file yet. GDELT 2.0 (doc's alternative news feed) is not evaluated.

In [ ]:
# ── EIA Open Data adapter (API v2) ────────────────────────────────────────────────
# Not a pre-existing aieng adapter -- built here against eia-api-swagger.yaml
# (route: /v2/petroleum/{route1}/{route2}/data, params: api_key, frequency,
# data[], facets[series][], sort[0][column]/[direction], offset, length).
# Mirrors FREDAdapter's cache-to-parquet convention.

EIA_BASE_URL = "https://api.eia.gov/v2"
EIA_PAGE_LENGTH = 5000  # API max rows per request


class EIAAdapter(BaseAdapter):
    """Adapter for a single EIA Open Data API v2 petroleum series, with disk cache.

    Parameters
    ----------
    route1, route2 : str
        EIA route segments, e.g. ``"pri", "spt"`` for spot prices.
    series_id : str
        EIA facet series id, e.g. ``"RWTC"`` or ``"EER_EPJK_PF4_RGC_DPG"``.
    frequency : str
        EIA frequency string (``"daily"``, ``"weekly"``, ...).
    api_key : str or None
        EIA API key. Defaults to the ``EIA_API_KEY`` environment variable.
    cache_dir : Path or None
        Parquet cache directory. Defaults to ``data/eia``.
    refresh : bool
        Force a network re-fetch even if a cache file exists.
    """

    def __init__(
        self,
        route1: str,
        route2: str,
        series_id: str,
        *,
        frequency: str = "daily",
        api_key: str | None = None,
        cache_dir: Path | None = None,
        refresh: bool = False,
    ) -> None:
        self._route1 = route1
        self._route2 = route2
        self._series_id = series_id
        self._frequency = frequency
        self._api_key = api_key or os.environ.get("EIA_API_KEY")
        self._cache_dir = cache_dir if cache_dir is not None else DATA_DIR / "eia"
        self._refresh = refresh

    @property
    def cache_path(self) -> Path:
        return self._cache_dir / f"{self._series_id}.parquet"

    def fetch(self) -> pd.DataFrame:
        if self.cache_path.exists() and not self._refresh:
            df = pd.read_parquet(self.cache_path)
            df["timestamp"] = pd.to_datetime(df["timestamp"])
            df["released_at"] = pd.to_datetime(df["released_at"])
            return df

        df = self._fetch_from_api()
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(self.cache_path, index=False)
        return df

    def _fetch_from_api(self) -> pd.DataFrame:
        if not self._api_key:
            raise ValueError(
                "EIA API key not provided. Set the EIA_API_KEY environment variable "
                "or pass api_key= to EIAAdapter."
            )

        url = f"{EIA_BASE_URL}/petroleum/{self._route1}/{self._route2}/data/"
        rows: list[dict] = []
        offset = 0
        while True:
            params = {
                "api_key": self._api_key,
                "frequency": self._frequency,
                "data[0]": "value",
                "facets[series][]": self._series_id,
                "sort[0][column]": "period",
                "sort[0][direction]": "asc",
                "offset": offset,
                "length": EIA_PAGE_LENGTH,
            }
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            payload = resp.json()["response"]
            page = payload["data"]
            rows.extend(page)
            if len(page) < EIA_PAGE_LENGTH:
                break
            offset += EIA_PAGE_LENGTH

        if not rows:
            raise RuntimeError(f"EIA series '{self._series_id}' returned no data.")

        df = pd.DataFrame(rows)
        df["timestamp"] = pd.to_datetime(df["period"])
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df = df.dropna(subset=["value"]).sort_values("timestamp").reset_index(drop=True)
        df["released_at"] = df["timestamp"]  # EIA doesn't expose vintage dates via this route
        return df[["timestamp", "value", "released_at"]]

    def __repr__(self) -> str:
        return f"EIAAdapter(route={self._route1}/{self._route2}, series_id={self._series_id!r})"


print("EIAAdapter defined.")

In [ ]:
# ── Canonical series IDs ───────────────────────────────────────────────────────────────
FUEL_SERIES_ID = "jet_fuel_proxy_price"  # yfinance HO=F -- primary target
WTI_SERIES_ID = "wti_crude_oil_price"  # yfinance CL=F -- covariate
USD_INDEX_SERIES_ID = "usd_index_price"  # yfinance DX-Y.NYB -- covariate
CAD_USD_FX_SERIES_ID = "cad_usd_fx_rate"  # FRED DEXCAUS -- covariate
EIA_WTI_SPOT_SERIES_ID = "eia_wti_spot_price"  # EIA RWTC -- covariate
EIA_JET_FUEL_SPOT_SERIES_ID = "eia_jet_fuel_spot_price"  # EIA EER_EPJK_PF4_RGC_DPG -- covariate
SHOCK_EVENT_SERIES_ID = "fuel_cost_shock_event_21d"  # derived binary series -- primary task target

_FUEL_TICKER = "HO=F"
_WTI_TICKER = "CL=F"
_USD_INDEX_TICKER = "DX-Y.NYB"
_HISTORY_START = "2004-01-01"


def derive_shock_event_series(
    price_df: pd.DataFrame,
    *,
    horizon_days: int = SHOCK_HORIZON_DAYS,
    threshold_pct: float = SHOCK_THRESHOLD_PCT,
) -> pd.DataFrame:
    """Derive the rolling binary cost-shock event series from a daily price series.

    For each row at date ``t``, looks ``horizon_days`` trading rows ahead to date
    ``t+h`` and sets ``value = 1.0`` if ``price[t+h] / price[t] - 1 > threshold_pct``,
    else ``0.0``. ``released_at`` is set to ``t+h`` -- the event is only knowable
    once the horizon has resolved -- so ``CutoffEnforcer`` correctly hides it from
    any origin before that date.
    """
    df = price_df.sort_values("timestamp").reset_index(drop=True)
    timestamps = pd.to_datetime(df["timestamp"])
    values = df["value"].astype(float)

    n = len(df)
    if n <= horizon_days:
        return pd.DataFrame(columns=["timestamp", "value", "released_at"])

    origin = values.iloc[: n - horizon_days].reset_index(drop=True)
    resolved = values.iloc[horizon_days:].reset_index(drop=True)
    pct_change = resolved / origin - 1.0

    return pd.DataFrame(
        {
            "timestamp": timestamps.iloc[: n - horizon_days].reset_index(drop=True),
            "value": (pct_change > threshold_pct).astype(float),
            "released_at": timestamps.iloc[horizon_days:].reset_index(drop=True),
        }
    )


class FuelShockEventAdapter(BaseAdapter):
    """Adapter producing the derived rolling cost-shock event series.

    Wraps the underlying price adapter and recomputes the derived series at
    fetch time, so it always reflects the freshest cached price data.
    """

    def __init__(self, price_adapter: BaseAdapter, *, horizon_days: int = SHOCK_HORIZON_DAYS, threshold_pct: float = SHOCK_THRESHOLD_PCT) -> None:
        self._price_adapter = price_adapter
        self._horizon_days = horizon_days
        self._threshold_pct = threshold_pct

    def fetch(self) -> pd.DataFrame:
        price_df = self._price_adapter.fetch()
        return derive_shock_event_series(price_df, horizon_days=self._horizon_days, threshold_pct=self._threshold_pct)


def build_fuel_service(cache_dir: Path | None = None) -> DataService:
    """Return a :class:`DataService` with the fuel cost-shock series registered.

    Registers the jet-fuel proxy target, the derived shock-event series, and
    the WTI / USD-index / CAD-USD-FX / EIA-spot covariates.
    """
    resolved_cache_dir = cache_dir if cache_dir is not None else DATA_DIR / "yfinance"
    svc = DataService()

    fuel_adapter = YFinanceDailyAdapter(ticker=_FUEL_TICKER, start=_HISTORY_START, cache_dir=resolved_cache_dir)
    svc.register(
        FUEL_SERIES_ID,
        fuel_adapter,
        SeriesMetadata(
            series_id=FUEL_SERIES_ID,
            description="NY Harbor ULSD / Heating Oil front-month futures adjusted close (Yahoo Finance HO=F) -- jet-fuel proxy.",
            source="yfinance",
            units="USD/gal",
            frequency="B",
        ),
    )

    svc.register(
        SHOCK_EVENT_SERIES_ID,
        FuelShockEventAdapter(fuel_adapter),
        SeriesMetadata(
            series_id=SHOCK_EVENT_SERIES_ID,
            description=(
                f"Cost-shock indicator: 1.0 if {FUEL_SERIES_ID} rises more than "
                f"{SHOCK_THRESHOLD_PCT:.0%} over the following {SHOCK_HORIZON_DAYS} business days, else 0.0."
            ),
            source=f"Derived ({FUEL_SERIES_ID})",
            units="0/1 event indicator",
            frequency="B",
        ),
    )

    svc.register(
        WTI_SERIES_ID,
        YFinanceDailyAdapter(ticker=_WTI_TICKER, start=_HISTORY_START, cache_dir=resolved_cache_dir),
        SeriesMetadata(
            series_id=WTI_SERIES_ID,
            description="WTI Crude Oil front-month futures adjusted close (Yahoo Finance CL=F)",
            source="yfinance",
            units="USD/bbl",
            frequency="B",
        ),
    )

    svc.register(
        USD_INDEX_SERIES_ID,
        YFinanceDailyAdapter(ticker=_USD_INDEX_TICKER, field="Adj Close", start=_HISTORY_START, cache_dir=resolved_cache_dir),
        SeriesMetadata(
            series_id=USD_INDEX_SERIES_ID,
            description="US Dollar Index close level (Yahoo Finance DX-Y.NYB)",
            source="yfinance",
            units="index-level",
            frequency="B",
        ),
    )

    try:
        svc.register(
            CAD_USD_FX_SERIES_ID,
            FREDAdapter("DEXCAUS", cache_dir=DATA_DIR / "fred"),
            SeriesMetadata(
                series_id=CAD_USD_FX_SERIES_ID,
                description="Canadian Dollars to U.S. Dollar spot exchange rate (FRED DEXCAUS)",
                source="FRED (DEXCAUS)",
                units="CAD per USD",
                frequency="B",
            ),
        )
    except (RuntimeError, ValueError) as exc:
        warnings.warn(f"Skipping CAD/USD FX (FRED): {exc}", stacklevel=2)

    try:
        svc.register(
            EIA_WTI_SPOT_SERIES_ID,
            EIAAdapter("pri", "spt", "RWTC"),
            SeriesMetadata(
                series_id=EIA_WTI_SPOT_SERIES_ID,
                description="Cushing, OK WTI spot price FOB (EIA RWTC)",
                source="EIA Open Data (petroleum/pri/spt, RWTC)",
                units="USD/bbl",
                frequency="B",
            ),
        )
        svc.register(
            EIA_JET_FUEL_SPOT_SERIES_ID,
            EIAAdapter("pri", "spt", "EER_EPJK_PF4_RGC_DPG"),
            SeriesMetadata(
                series_id=EIA_JET_FUEL_SPOT_SERIES_ID,
                description="US Gulf Coast kerosene-type jet-fuel spot price FOB (EIA EER_EPJK_PF4_RGC_DPG)",
                source="EIA Open Data (petroleum/pri/spt, EER_EPJK_PF4_RGC_DPG)",
                units="USD/gal",
                frequency="B",
            ),
        )
    except (RuntimeError, ValueError) as exc:
        warnings.warn(f"Skipping EIA spot-price covariates: {exc}", stacklevel=2)

    return svc


print("build_fuel_service() defined.")

In [ ]:
# Build the service and preview the price history through today.
data_service = build_fuel_service()
ctx0 = data_service.context(as_of=naive_utc_now())
fuel_df = ctx0.get_series(FUEL_SERIES_ID)
print(f"Jet-fuel proxy (HO=F) history through {pd.Timestamp(fuel_df['timestamp'].max()).date()}, {len(fuel_df)} rows")
print(f"Registered series: {list(data_service.series_ids)}")

---
## 2. Geopolitical / News Agent

A `ContextRetrievalConfig` sub-agent (the same primitive `energy_oil_forecasting` uses for its news-grounded configs) with a temporal-leakage verifier, wired into the Forecaster Agent below as the `search_web` tool. GDELT 2.0 (doc's alternative feed) is not wired up here.

In [ ]:
FUEL_NEWS_INSTRUCTION = """
You are a jet-fuel and crude-oil market intelligence specialist with access to web search.

Search for information relevant to the query and return a concise structured markdown summary (3-5 paragraphs) covering relevant aspects of:
- Crude oil (WTI/Brent) and refined-products (jet fuel / heating oil / diesel) price level and recent trend
- OPEC+ production decisions and supply outlook
- Geopolitical risks in the Persian Gulf, Middle East, Strait of Hormuz, and other key shipping lanes affecting crude and product tanker flows
- US EIA policy releases, Strategic Petroleum Reserve actions, and inventory data surprises
- Refinery outages or disruptions affecting jet fuel / distillate supply
- Notable analyst forecasts or unusual price-target revisions for crude or refined products

Ground your summary in the search results you actually retrieve. When a cutoff date is specified, do not report or speculate about events that occurred after that date.

Before finalizing your summary, reason step by step: (1) for each candidate fact, judge its actual recency from the substance of the result itself, never from a source's claimed publish date or byline timestamp -- those are frequently stale or updated after original publication; (2) discard anything you cannot confidently place before the cutoff date; (3) only then write your summary. Do not supplement the search results with your own background/training knowledge -- if the results are insufficient, say so explicitly rather than filling gaps from memory.
""".strip()

FUEL_CONTEXT_RETRIEVAL_SUPPLEMENT = """

## Context retrieval

Call ``search_web`` to gather market intelligence BEFORE producing forecasts.

Call ``search_web`` with ``query`` and ``cutoff_date`` (set to the ``as_of`` date from the payload). The ``cutoff_date`` MUST always equal ``as_of`` -- this is the temporal fence that prevents post-origin information from contaminating historical backtests.

If ``search_web`` returns a result beginning with ``[SEARCH_VERIFICATION_FAILED]``, treat it as no verified news context for that query. Do not use your own background knowledge to fill the gap or speculate about what the news might have said -- proceed with price-history and other available signals only, and note the gap in your rationale.

Recommended queries (call ``search_web`` once per topic):
- ``search_web(query="crude oil and jet fuel price trend and OPEC+ supply decisions", cutoff_date=<as_of>)``
- ``search_web(query="Persian Gulf geopolitical risk shipping lane disruptions", cutoff_date=<as_of>)``
- ``search_web(query="US EIA policy releases and Strategic Petroleum Reserve actions", cutoff_date=<as_of>)``
- ``search_web(query="refinery outages affecting jet fuel and distillate supply", cutoff_date=<as_of>)``
"""


def build_fuel_context_retrieval_config(
    search_model: str = SEARCH_MODEL,
    verifier_model: str = VERIFIER_MODEL,
    verifier_max_attempts: int = 3,
    verifier_confidence_threshold: int = 8,
) -> ContextRetrievalConfig:
    """Build the Geopolitical/News Agent's :class:`ContextRetrievalConfig`."""
    return ContextRetrievalConfig(
        enabled=True,
        instruction=FUEL_NEWS_INSTRUCTION,
        search_model=search_model,
        verifier_model=verifier_model,
        verifier_max_attempts=verifier_max_attempts,
        verifier_confidence_threshold=verifier_confidence_threshold,
    )


print("News Agent instruction + factory defined.")

---
## 3. Forecaster Agent

"One agent, two tasks": a single identity, with the ask living in each call's `task_spec` (not the system instruction). `FuelMultitaskPromptBuilder` builds the payload from the Commodity-Data service above.

In [ ]:
FUEL_FORECASTER_INSTRUCTION = """
## Role

You are an expert jet-fuel and crude-oil market analyst producing calibrated forecasts to support airline hedging, budgeting, and financial risk decisions.

## Input

You will receive a JSON payload containing:
- `task_spec`: the exact question and required JSON output schema
- `as_of`: the forecast origin date (temporal cutoff)
- `horizons`: integer horizon steps (business days ahead)
- `standard_quantiles`: quantile levels for continuous forecasts (when applicable)
- `origin_price_usd_gal`: jet-fuel proxy (heating oil futures) close on the origin date
- `target_history_csv`: compressed jet-fuel proxy daily close history

Call ``search_web`` BEFORE answering -- it is your only source of qualitative market intelligence (OPEC+ policy, geopolitical supply risk, EIA releases); you have no other tools and no post-training-cutoff knowledge to rely on.

## Output contract

Read the data and the search briefing carefully, then execute the task in `task_spec` precisely.

If a `set_model_response` tool is available, call it with your complete JSON as `json_response` -- the exact schema is described in `task_spec`. Otherwise return the JSON directly as plain text with no preamble.
""".strip()


def build_fuel_forecaster_config(
    model: str = AGENT_MODEL,
    search_model: str = SEARCH_MODEL,
    verifier_model: str = VERIFIER_MODEL,
) -> AgentConfig:
    """Build the Forecaster Agent's :class:`AgentConfig`, wiring in the News Agent."""
    return AgentConfig(
        name="fuel_shock_forecaster",
        model=model,
        instruction=FUEL_FORECASTER_INSTRUCTION,
        context_retrieval=build_fuel_context_retrieval_config(search_model=search_model, verifier_model=verifier_model),
    )


def _compress_history(df: pd.DataFrame) -> str:
    """Compress daily price history: recent 6 months daily, older weekly averages."""
    frame = df.copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"])
    cutoff = frame["timestamp"].max() - pd.DateOffset(months=6)

    recent = frame[frame["timestamp"] >= cutoff]
    old = frame[frame["timestamp"] < cutoff]

    rows: list[str] = ["date,close"]
    if not old.empty:
        weekly = old.set_index("timestamp")["value"].resample("W").mean().dropna()
        rows.extend(f"{date.date()},{val:.3f}" for date, val in weekly.items())
    rows.extend(f"{row['timestamp'].date()},{row['value']:.3f}" for _, row in recent.iterrows())
    return "\n".join(rows)


class FuelMultitaskPromptBuilder(BaseModel):
    """Prompt builder for task-spec-driven Forecaster Agent calls.

    The system instruction is task-agnostic; the ask lives in ``task_spec``.
    """

    task_spec: str

    model_config = {"extra": "forbid"}

    def __call__(self, *, task: ForecastingTask, context: ForecastContext) -> str:
        df = context.get_series(task.target_series_id)
        last_row = df.iloc[-1]
        payload = {
            "task": task.task_id,
            "task_spec": self.task_spec,
            "as_of": str(context.as_of)[:10],
            "horizons": list(task.horizons),
            "standard_quantiles": list(STANDARD_QUANTILES),
            "origin_price_usd_gal": float(last_row["value"]),
            "target_history_csv": _compress_history(df),
        }
        return json.dumps(payload, indent=2)


TASK_SHOCK_SPEC = (
    f"Estimate the probability that the jet-fuel proxy price will close MORE\n"
    f"THAN {SHOCK_THRESHOLD_PCT:.0%} HIGHER than today's price at the end of\n"
    f"{SHOCK_HORIZON_DAYS} trading days (the cost-shock event).\n\n"
    "This is a directional upside question only.\n\n"
    "Calibration guidance:\n"
    "  - No unusual upside catalyst       -> base rate ~10-15%\n"
    "  - Escalating unconfirmed risk      -> 20-40%\n"
    "  - Confirmed supply disruption      -> 60-85%\n\n"
    "Use `key_signals` to name the specific drivers (OPEC+ decisions, shipping-lane\n"
    "risk, inventory levels, refinery outages, macro demand) behind your estimate --\n"
    "this is the primary place downstream readers look for the 'why'.\n\n"
    "If a `set_model_response` tool is available, call it with your complete "
    "JSON as `json_response`. Otherwise return the JSON directly as plain text.\n\n"
    "Required JSON format:\n" + DiscreteAgentForecastOutput.prompt_schema_json()
)

TASK_TRAJECTORY_SPEC = (
    "Forecast the jet-fuel proxy price at each horizon listed in the payload "
    "(`horizons`, business days ahead).\n\n"
    "Rules:\n"
    "  - Produce one forecast for each horizon in `horizons`.\n"
    "  - Use exactly the quantile levels from `standard_quantiles` -- no additions, no omissions.\n"
    "  - `point_forecast` must exactly equal the 0.50 quantile value.\n"
    "  - Quantile values must be strictly non-decreasing as quantile levels increase.\n"
    "  - Document your reasoning in the `rationale` fields.\n\n"
    "If a `set_model_response` tool is available, call it with your complete "
    "JSON as `json_response`. Otherwise return the JSON directly as plain text.\n\n"
    "Required JSON format:\n" + ContinuousAgentForecastOutput.prompt_schema_json()
)

forecaster_config = build_fuel_forecaster_config()
print(f"Forecaster Agent config: {forecaster_config.name} (model={forecaster_config.model})")

---
## 4. Run the pipeline

Wires the Forecaster Agent to each task and runs it at a single live origin. Set `USE_CACHE = False` (default) after any edits above.

In [ ]:
USE_CACHE = False
SHOCK_CACHE = DATA_DIR / "fuel_shock_forecast.json"
TRAJ_CACHE = DATA_DIR / "fuel_trajectory_forecast.json"

shock_task = ForecastingTask(
    task_id="fuel_cost_shock_next_month",
    target_series_id=FUEL_SERIES_ID,
    horizons=[SHOCK_HORIZON_DAYS],
    frequency="B",
    description="Binary cost-shock demo",
)
shock_prompt_builder = FuelMultitaskPromptBuilder(task_spec=TASK_SHOCK_SPEC)
shock_predictor = AgentPredictor(
    agent_config=forecaster_config,
    prompt_builder=shock_prompt_builder,
    output_schema=DiscreteAgentForecastOutput,
)

trajectory_task = ForecastingTask(
    task_id="fuel_trajectory_next_month",
    target_series_id=FUEL_SERIES_ID,
    horizons=list(TRAJECTORY_HORIZONS),
    frequency="B",
    description="Trajectory demo",
)
trajectory_prompt_builder = FuelMultitaskPromptBuilder(task_spec=TASK_TRAJECTORY_SPEC)
trajectory_predictor = AgentPredictor(
    agent_config=forecaster_config,
    prompt_builder=trajectory_prompt_builder,
    output_schema=ContinuousAgentForecastOutput,
)

print(f"Shock predictor schema: {shock_predictor.output_schema.__name__}")
print(f"Trajectory predictor schema: {trajectory_predictor.output_schema.__name__}")

In [ ]:
# Run the shock task at today's origin (live -- no historical temporal fence).
if USE_CACHE and SHOCK_CACHE.exists():
    with open(SHOCK_CACHE) as f:
        shock_result = json.load(f)
    print("Loaded cached shock forecast.")
else:
    live_ctx = data_service.context(as_of=naive_utc_now())
    preds = shock_predictor.predict(shock_task, live_ctx)
    shock_result = preds[0].model_dump(mode="json")
    with open(SHOCK_CACHE, "w") as f:
        json.dump(shock_result, f, indent=2)
    print("Saved shock forecast.")

payload = shock_result["payload"]
meta = shock_result.get("metadata", {})
display(
    Markdown(
        f"### Cost-shock forecast\n\n"
        f"**P(> {SHOCK_THRESHOLD_PCT:.0%} rise in {SHOCK_HORIZON_DAYS} trading days) = {payload['probability']:.0%}**\n\n"
        f"- Direction bias: {payload.get('direction_bias', '?')}\n"
        f"- Confidence: {payload.get('confidence', '?')}\n"
        f"- Key signals: {', '.join(payload.get('key_signals', [])) or '—'}\n\n"
        f"> {payload.get('reasoning', '')}"
    )
)

In [ ]:
# Run the trajectory task at today's origin.
if USE_CACHE and TRAJ_CACHE.exists():
    with open(TRAJ_CACHE) as f:
        traj_result = json.load(f)
    print("Loaded cached trajectory forecast.")
else:
    live_ctx = data_service.context(as_of=naive_utc_now())
    preds = trajectory_predictor.predict(trajectory_task, live_ctx)
    traj_result = [p.model_dump(mode="json") for p in preds]
    with open(TRAJ_CACHE, "w") as f:
        json.dump(traj_result, f, indent=2)
    print("Saved trajectory forecast.")

for h, pred in zip(TRAJECTORY_HORIZONS, traj_result, strict=False):
    pt = pred["payload"]["point_forecast"]
    print(f"  h={h}bd  point_forecast=${pt:.3f}/gal")

---
## 5. Evaluation helpers

Reuses `compute_brier_score` from `aieng.forecasting.evaluation` (the doc's "Evaluation Framework: reused" item) for the binary shock task. See `specs/fuel_shock_smoke.yaml` / `specs/fuel_shock_backtest.yaml` for the full backtest harness runs.

**Baselines (doc §7) not yet implemented:** `aieng.forecasting.methods` only has naive baselines today. GARCH and logistic regression need to be added before a leaderboard comparison is possible.

In [ ]:
def shock_calibration_table(probabilities: list[float], outcomes: list[float], n_bins: int = 5) -> pd.DataFrame:
    """Bin predicted probabilities and compare to realised outcome frequency (reliability diagram data)."""
    df = pd.DataFrame({"probability": probabilities, "outcome": outcomes}).dropna()
    if df.empty:
        return pd.DataFrame(columns=["bin", "mean_predicted", "mean_observed", "n"])
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    df["bin"] = pd.cut(df["probability"], bins=edges, include_lowest=True)
    grouped = df.groupby("bin", observed=True).agg(
        mean_predicted=("probability", "mean"), mean_observed=("outcome", "mean"), n=("outcome", "count")
    )
    return grouped.reset_index()


print("compute_brier_score / shock_calibration_table ready -- use with a multi-origin backtest run (see specs/).")